# Vectorization

Section 13.2 of the notes.

An operation on whole arrays runs one loop written in C. The same operation
written as a Python loop enters the interpreter once per entry. This notebook
measures the difference on three problems: a dot product, a group-wise sum, and
the second-difference stencil of Chapter 2.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Timings use %timeit -o -q:
# -o returns the result instead of only printing it,
# -q keeps it quiet, and .best is the fastest run, divided by the loops per run.

## 1. A dot product

In [ ]:
def dot_loop(a, b):
    s = 0.0
    for i in range(a.size):
        s += a[i] * b[i]
    return s


n = 1_000_000
a, b = rng.standard_normal(n), rng.standard_normal(n)

t_loop = %timeit -o -q dot_loop(a, b)
t_vec = %timeit -o -q a @ b
t_loop, t_vec = t_loop.best, t_vec.best

print(f"loop  {1e9 * t_loop / n:7.2f} ns per entry")
print(f"a @ b {1e9 * t_vec / n:7.3f} ns per entry")
print(f"ratio {t_loop / t_vec:7.0f}")
print("same result:", abs(dot_loop(a, b) - a @ b) < 1e-6 * abs(a @ b))

The results differ in the last digits, because the additions happen in a
different order. Comparisons of floating point results use a tolerance.

## 2. Group-wise sums

There are two arrays of length $N$: `values`, the data, and `labels`, where
`labels[i]` is the number, from $0$ to $K-1$, of the group that `values[i]`
belongs to (for instance the grid cell in which measurement $i$ was taken). For
every group $g$ we want the sum of the values with `labels[i] == g`, and how
many there are; their ratio is the mean of the group.

A small example with $K=3$ groups:

```
labels = [2, 0, 2, 1, 0]
values = [1.0, 4.0, 3.0, 5.0, 2.0]
sums   = [4.0 + 2.0, 5.0, 1.0 + 3.0] = [6.0, 5.0, 4.0]
counts = [2, 1, 2]
```

The loop visits the values one by one and adds each to the running total of its
group. `np.bincount(labels)` does the counting in one call: entry $g$ of the
result is the number of times $g$ occurs in `labels`. With `weights=values` it
adds up the values instead of counting them, so entry $g$ is the sum of the
values labelled $g$. `minlength=K` keeps an entry for every group, even for a
group whose label never occurs.

In [ ]:
N, K = 2_000_000, 5_000
labels = rng.integers(0, K, size=N, dtype=np.int64)
values = rng.standard_normal(N)


def groups_loop():
    sums, counts = {}, {}  # group label -> running total, count
    for g, v in zip(labels, values):
        sums[g] = sums.get(g, 0.0) + v
        counts[g] = counts.get(g, 0) + 1
    s = np.zeros(K)  # copy into arrays, as bincount returns
    c = np.zeros(K, dtype=np.int64)
    for g, total in sums.items():
        s[g], c[g] = total, counts[g]
    return s, c


def groups_bincount():
    c = np.bincount(labels, minlength=K)
    s = np.bincount(labels, weights=values, minlength=K)
    return s, c

In [ ]:
s_loop, c_loop = groups_loop()
s_vec, c_vec = groups_bincount()
print("same sums:  ", np.allclose(s_loop, s_vec))
print("same counts:", np.array_equal(c_loop, c_vec))

t_loop = %timeit -o -q groups_loop()
t_vec = %timeit -o -q groups_bincount()
t_loop, t_vec = t_loop.best, t_vec.best
print(f"loop      {t_loop * 1e3:8.1f} ms")
print(f"bincount  {t_vec * 1e3:8.1f} ms")
print(f"ratio     {t_loop / t_vec:8.0f}")

The means follow from the two arrays, with `where` to leave the empty groups at
zero instead of dividing by zero.

In [ ]:
means = np.divide(s_vec, c_vec, out=np.zeros(K), where=c_vec > 0)
print("groups without any value:", int((c_vec == 0).sum()))
print("first five means:", means[:5])

## 3. The stencil of Chapter 2

The method of lines needs the periodic second difference

$$ (u_{j+1} - 2u_j + u_{j-1}) / \Delta x^2 . $$

Three versions: a Python loop, `np.roll`, and slices that write into an array
that already exists. The slices are views, so the third version allocates
nothing at all.

In [ ]:
def stencil_loop(u, out, dx):
    d = u.size
    for j in range(d):
        out[j] = (u[(j + 1) % d] - 2 * u[j] + u[j - 1]) / dx**2
    return out


def stencil_roll(u, dx):
    return (np.roll(u, -1) - 2 * u + np.roll(u, 1)) / dx**2


def stencil_slices(u, out, dx):
    out[1:-1] = (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2
    out[0] = (u[1] - 2 * u[0] + u[-1]) / dx**2
    out[-1] = (u[0] - 2 * u[-1] + u[-2]) / dx**2
    return out

In [ ]:
d = 2048  # the grid of the reference solution in Chapter 2
dx = 2 / d
u = np.cos(np.pi * np.linspace(-1, 1, d, endpoint=False))
out = np.empty_like(u)

r_loop = stencil_loop(u, np.empty_like(u), dx)
r_roll = stencil_roll(u, dx)
r_slice = stencil_slices(u, out, dx)
print("all three agree:", np.allclose(r_loop, r_roll) and np.allclose(r_loop, r_slice))
print("np.roll allocates:", not np.shares_memory(np.roll(u, -1), u))

t_loop = %timeit -o -q stencil_loop(u, out, dx)
t_roll = %timeit -o -q stencil_roll(u, dx)
t_slice = %timeit -o -q stencil_slices(u, out, dx)
t_loop, t_roll, t_slice = t_loop.best, t_roll.best, t_slice.best
for name, t in [("loop", t_loop), ("np.roll", t_roll), ("slices", t_slice)]:
    print(f"{name:8s} {1e9 * t / d:7.2f} ns per grid point")

The gap between the two array versions narrows as the grid grows. At this
size the fixed cost of allocating the two shifted arrays still dominates,
while at $10^6$ points and beyond both are limited by the memory and the
difference is about twenty per cent.

## 4. Temporaries

One forward Euler step of the bistable equation $du/dt = u - u^3$ (Example 1.10),
applied to every entry of an array `u`, is `u + dt * (u - u**3)`. NumPy computes
it one operation at a time, and each operation stores its result in a new
array, a *temporary*:

1. `u**3` is a new array;
2. `u - u**3` is a second new array. NumPy writes a result into a temporary
   that is no longer needed when it can, but for a subtraction only into the
   one on the left of the minus, and here the temporary is on the right;
3. `dt * (...)` and `u + (...)` reuse the array of step 2.

At the peak three arrays are alive: `u`, `u**3`, and `u - u**3`.

The second version overwrites `u`, which is no longer needed once the step is
done, and uses one scratch array `tmp`, allocated once before the time loop and
passed to every step. With `out=` every operation writes into `tmp` or `u`, so
only these two arrays are ever alive.

The cell runs a few steps of both versions on 50 million entries, 400 MB per
array. Both start from a copy of the same `u0`, made inside the measurement, so
the memory counted includes `u` and `tmp`.

In [ ]:
import tracemalloc


def step_temporaries(u, dt):
    # u**3 and u - u**3 are new arrays; the new u is returned as a new array
    return u + dt * (u - u**3)


def step_in_place(u, dt, tmp):
    # every result goes into tmp or u, so nothing new is allocated
    np.power(u, 3, out=tmp)  # tmp = u**3
    np.subtract(u, tmp, out=tmp)  # tmp = u - u**3
    np.multiply(tmp, dt, out=tmp)  # tmp = dt * (u - u**3)
    np.add(u, tmp, out=u)  # u = u + tmp, overwriting u


def run_temporaries(u0, dt, steps):
    u = u0.copy()
    for _ in range(steps):
        u = step_temporaries(u, dt)
    return u


def run_in_place(u0, dt, steps):
    u = u0.copy()
    tmp = np.empty_like(u0)  # allocated once, reused by every step
    for _ in range(steps):
        step_in_place(u, dt, tmp)
    return u


def peak_memory(run, *args):
    # largest amount of memory, in bytes, allocated at one time during run(*args)
    tracemalloc.start()
    run(*args)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak


u0 = rng.random(50_000_000)  # 400 MB
dt, steps = 0.01, 5

print(
    "same result:",
    np.allclose(run_temporaries(u0, dt, steps), run_in_place(u0, dt, steps)),
)

for name, run in [("temporaries", run_temporaries), ("in place", run_in_place)]:
    t = %timeit -o -q -r 3 -n 1 run(u0, dt, steps)
    print(
        f"{name:12s} {1e3 * t.best / steps:6.1f} ms per step,"
        f" peak memory {peak_memory(run, u0, dt, steps) / 1e6:5.0f} MB"
    )

The version in place needs two arrays instead of three, a third less memory,
and takes the same time: both are limited by how fast memory is read and
written. The saving comes from overwriting `u`. Writing into arrays allocated
beforehand saves memory only when they replace temporaries that would be alive
at the same time.